# NeuralAI D17 — Colab DPO Training
Runs the existing `train_d17_dpo.py` on a Colab GPU.

- Base: `HuggingFaceTB/SmolLM2-360M-Instruct`
- Reference: v16 adapter at `checkpoints/v2_model` (checkpoint-69, best)
- Data: `data/train_dpo_v16_combined.jsonl`
- Output: `checkpoints/v17-dpo`

**Set runtime to GPU (T4 or better) before running.**

In [ ]:
# @title 1. Mount Drive (optional — only if your repo/assets live there)
# If everything is in the GitHub repo, skip this and go to cell 2.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. Clone repo + pin GPU
import os, subprocess, torch

REPO = "https://github.com/Subject-Emu-5259/NeuralAI.git"
WORK = "/content/NeuralAI"

if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", REPO, WORK], check=True)
else:
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)

os.chdir(WORK)
print("cwd:", os.getcwd())
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("NO GPU — switch runtime to GPU (T4) before running.")

In [ ]:
# @title 3. Install deps (pinned, Colab 2026-compatible)
subprocess.run(["pip", "install", "-q",
    "transformers==4.51.3",
    "peft==0.15.2",
    "trl==0.18.1",
    "bitsandbytes==0.45.5",
    "datasets==3.5.1",
    "accelerate==1.6.0",
    "huggingface_hub",
], check=True)
print("deps installed")

In [ ]:
# @title 4. Verify assets exist before training
for p in ["checkpoints/v2_model/adapter_model.safetensors",
          "checkpoints/v2_model/adapter_config.json",
          "data/train_dpo_v16_combined.jsonl",
          "train_d17_dpo.py"]:
    print(("OK  " if os.path.exists(p) else "MISSING "), p)
assert os.path.exists("train_d17_dpo.py"), "train_d17_dpo.py not found"

In [ ]:
# @title 5. Patch train_d17_dpo.py to force CUDA (Colab GPU)
# The script defaults to CPU when torch.cuda.is_available() is False.
# On Colab with a GPU we want CUDA, so we rewrite the device line + bnb device_map.
with open("train_d17_dpo.py") as f:
    src = f.read()

# Force CUDA device instead of cpu fallback
src = src.replace(
    'device = "cuda" if torch.cuda.is_available() else "cpu"',
    'device = "cuda"  # forced for Colab GPU'
)
# Keep base model on GPU (was device_map="cpu")
src = src.replace(
    'device_map="cpu"',
    'device_map="auto"'
)
# Enable bf16 on GPU (was gated on cuda availability — already true, keep)
with open("train_d17_dpo.py", "w") as f:
    f.write(src)
print("patched train_d17_dpo.py for CUDA")

In [ ]:
# @title 6. Run D17 DPO training
# This is long. Colab will stream logs. Expect several hundred steps.
subprocess.run(["python", "train_d17_dpo.py"], check=True)

In [ ]:
# @title 7. Upload v17-dpo adapter to HuggingFace
# Set your HF token in Colab Secrets (name: HF_TOKEN) or paste below.
from huggingface_hub import HfApi, login
import getpass

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF token: ")
login(token=token)

api = HfApi()
REPO_ID = "Subject-Emu-5259/NeuralAI"  # v17 adapter folder = v17-dpo
api.upload_folder(
    folder_path="checkpoints/v17-dpo",
    repo_id=REPO_ID,
    path_in_repo="v17-dpo",
)
print("uploaded checkpoints/v17-dpo ->", REPO_ID + "/v17-dpo")